# Day 5.1 — What Is an AI Harness?

For four days we built one agent at a time. Each project quietly repeated the same
chores: pick a model, describe the tools, run a loop, check whether an action was
allowed, remember something, print what happened.

A **runtime** is the code that executes one run. A **harness** is the runtime plus
everything around it — configuration, tool registry, policy, events, checkpoints.
This lesson names those repeated chores and shows you where each one now lives.


## Before you begin

### Learning outcomes

- Name the responsibilities every Day 1-4 project repeated.
- Point at the mini-harness module that now owns each one.
- Tell an agent, a framework, a protocol, a runtime and a harness apart.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Two different agent configurations print different behaviour while pointing at the same runtime modules. A printed table links each Day 1-4 pain point to one component.


## Concept briefing

## Why consolidate the earlier projects

By Day 5, several applications repeat the same responsibilities: load model
configuration, describe tools, validate arguments, enforce policy, limit steps, record
events and save continuation state. Copying this code into every agent makes safety fixes
inconsistent. A reusable runtime centralises the execution lifecycle.

This course uses **harness** as an umbrella term for the environment around an agent. In
industry, related terms include agent runtime, orchestration layer and agent platform.
The exact vocabulary varies; the responsibilities are transferable.


### Before you run: the `.env` file

Day 5 runs end to end with **no API key**. If you want the optional live cells, the key
lives in a file called `.env` at the repository root (next to `README.md`), containing
`OPENROUTER_API_KEY=sk-or-...` and `OPENROUTER_MODEL=openai/gpt-oss-120b`.
**Day 1.1 walks through creating it.** Without it the setup cell below simply reports
`MOCK` and every lesson still works.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — Two agents, described as data

An agent configuration is a JSON file, not code. It says who the agent is, which tools
it may see, and how many steps it gets. Load two of them and look at the difference.


In [ ]:
# Read both configurations from configs/*.json.
research = load_config("research_agent")
task = load_config("task_agent")

for config in (research, task):
    print("Agent name    :", config.name)
    print("  instructions:", config.instructions)
    print("  allowed tools:", config.allowed_tools)   # the allow-list, not "all tools"
    print("  max steps    :", config.max_steps)
    print("  model        :", config.model.provider, "/", config.model.model)
    print()

print("Two different agents. Zero lines of agent-specific Python so far.")

## Step 2 — The problem each harness component solves

Every component of the mini harness exists because something hurt on an earlier day.
This table is the whole point of Day 5, so we print it rather than describing it.


In [ ]:
# Left column: something that was awkward or unsafe in an earlier project.
# Right column: the mini-harness module that now owns it, once, for every agent.
LINEAGE = [
    ("Day 1", "every script rebuilt the API request by hand",
     "schemas.ModelConfig + providers.build_provider"),
    ("Day 1", "tools were if/elif branches inside the loop",
     "registry.ToolRegistry"),
    ("Day 2", "each project re-invented where retrieved state lived",
     "events.EventStore + events.JSONCheckpointStore"),
    ("Day 3", "safety checks were copy-pasted into each agent",
     "policy.decide"),
    ("Day 3", "approval meant 'ask in the chat and hope'",
     "runtime pause + checkpoint + runtime.resume"),
    ("Day 4", "a chatty team could loop and nobody could stop it",
     "runtime.MAX_STEPS_HARD_CAP"),
    ("Day 4", "explaining what happened meant re-reading print statements",
     "events.EventStore"),
]

width = max(len(problem) for _, problem, _ in LINEAGE)
print(f"{'Day':<6} {'Problem we actually hit':<{width}}  ->  Harness component")
print("-" * (width + 50))
for day, problem, component in LINEAGE:
    print(f"{day:<6} {problem:<{width}}  ->  {component}")

## Step 3 — The modules, and what each one refuses to do

Look at the package itself. Each module is small on purpose: you should be able to read
any one of them in a couple of minutes.


In [ ]:
import mini_harness

RESPONSIBILITIES = {
    "schemas":   "the data contracts (config, tool spec, decision, result)",
    "providers": "talk to a model; mock, OpenRouter or Ollama",
    "registry":  "hold tools, describe them, validate arguments",
    "policy":    "answer 'may this agent do this now?'",
    "runtime":   "run the loop, apply limits, pause for approval",
    "events":    "append-only record + resumable checkpoint",
    "memory":    "a deliberately tiny recall interface",
    "mcp_client": "discover capabilities from an outside server",
}
for module, job in RESPONSIBILITIES.items():
    print(f"mini_harness.{module:<11} : {job}")

print()
print("Public names the harness exports:", len(mini_harness.__all__))
print("Note what is NOT here: authentication, sandboxing, deployment, a UI.")
print("Those are real responsibilities - they are just not this course's subject.")

## Step 4 — Vocabulary, checked against the objects

These five words get used interchangeably online. Here they mean specific things, and
you can see each one as a Python object.


In [ ]:
from mini_harness import HarnessRuntime, MockModel, build_demo_registry

runtime = HarnessRuntime(build_demo_registry(), MockModel())

print("agent      :", research.name, "- a CONFIGURATION;", type(research).__name__)
print("framework  : the library you import; here, plain Python +", type(runtime).__name__)
print("protocol   : an interoperability contract, e.g. MCP (Day 5.6). Not shown yet.")
print("runtime    : the object that executes one run ->", type(runtime).__name__)
print("harness    : that runtime PLUS registry, policy, events, checkpoints")
print()
print("Same runtime object, two agents:")
print("  ", runtime.run(research, "What is a harness?").status,
      "<- research_agent")
print("  ", runtime.run(task, "Prepare a short update").status,
      "<- task_agent")

### Try it yourself

Before running the next cell, predict: a configuration asks for `max_steps = 500`.
How many steps will the runtime actually take?


In [ ]:
# --- Worked solution ---
# The configuration is a REQUEST. The runtime owns the ceiling, so the effective
# limit is min(config.max_steps, MAX_STEPS_HARD_CAP) - never the larger number.
from mini_harness import MAX_STEPS_HARD_CAP, effective_step_limit

greedy = load_config("research_agent")
greedy.max_steps = 500                      # a configuration can ask for anything

print("Hard cap written into runtime.py :", MAX_STEPS_HARD_CAP)
print("Steps this configuration requests:", greedy.max_steps)
print("Steps it will actually be given  :", effective_step_limit(greedy))
print()
print("A modest config is NOT raised to the cap:")
modest = load_config("research_agent")      # max_steps = 3 in the JSON file
print("  requested:", modest.max_steps, "-> effective:", effective_step_limit(modest))
print()
print("Lesson: limits belong to the runtime, not to the thing being limited.")

### Checkpoint

**1. What is the difference between an agent and a runtime?**

<details><summary>Show answer</summary>

An agent is a *configuration* - instructions, an allow-list of tools, limits, a model choice. It is data and lives in `configs/*.json`. A runtime is the *code* that executes that configuration. One runtime ran both agents in Step 4 without knowing anything about either of them.

</details>

**2. Why is `max_steps` in the agent config but the hard cap in `runtime.py`?**

<details><summary>Show answer</summary>

Because they answer different questions. `max_steps` is what this application thinks it needs; the hard cap is what the platform is willing to allow. If the ceiling lived in the config, any config could raise it, and the limit would stop being a limit.

</details>

### Recap

- Limitation: every Day 1-4 project re-implemented configuration, tools, policy, limits and logging, so a fix in one never reached the others.
- Layer added: a named module per responsibility, with agent behaviour pushed out into JSON configuration files.
- Evidence: one `HarnessRuntime` object ran two different agents, and a config asking for 500 steps was silently held to the runtime's cap of 10.
